# Imports

In [1]:
# Standard Import

import requests
from datetime import datetime

import numpy as np
import pandas as pd

import random
import math

import matplotlib.pyplot as plt

from IPython.display import clear_output

import time
import os

# ML Import

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor
import numpy as np

# Portfolio Management Import
from scipy.optimize import linprog
from scipy.optimize import minimize

# Functions

In [2]:
## Fetching data functions
def fetch_data(instrument: str) -> pd.DataFrame:
    url = "https://data-api.coindesk.com/spot/v1/historical/days"
    headers = {
        "accept": "application/json",
        "x-api-key": ""  # Replace with your actual API key
    }
    params = {
        "market": "bybit",
        "instrument": instrument,
        "limit": 200,
        "aggregate": 1,
        "fill": "true",
        "apply_mapping": "true",
        "response_format": "JSON"
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()  # Raise error if request failed

    data = response.json()
    if "Data" not in data or not data["Data"]:
        print("No data found for instrument:", instrument)
        return pd.DataFrame()

    ohlcv = [
        {
            "open": entry["OPEN"],
            "high": entry["HIGH"],
            "low": entry["LOW"],
            "close": entry["CLOSE"],
            "volume": entry["VOLUME"]
        }
        for entry in data["Data"]
    ]

    return pd.DataFrame(ohlcv)

# Fetch data on short range (30 time frame) only
# -> Covariance depends on RL situation
def fetch_data_short(instrument: str) -> pd.DataFrame:
    url = "https://data-api.coindesk.com/spot/v1/historical/days"
    headers = {
        "accept": "application/json",
        "x-api-key": ""  # Replace with your actual API key
    }
    params = {
        "market": "bybit",
        "instrument": instrument,
        "limit": 16,
        "aggregate": 1,
        "fill": "true",
        "apply_mapping": "true",
        "response_format": "JSON"
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()  # Raise error if request failed

    data = response.json()
    if "Data" not in data or not data["Data"]:
        print("No data found for instrument:", instrument)
        return pd.DataFrame()

    ohlcv = [
        {
            "open": entry["OPEN"],
            "high": entry["HIGH"],
            "low": entry["LOW"],
            "close": entry["CLOSE"],
            "volume": entry["VOLUME"]
        }
        for entry in data["Data"]
    ]

    return pd.DataFrame(ohlcv)

# Getting top coins list from API
def get_top_coins(vs_currency="usd"):
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": vs_currency,
        "order": "market_cap_desc",
        "per_page": 250,
        "page": 1,
        "sparkline": False
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch data: {response.status_code} - {response.text}")

    data = response.json()
    symbols = [coin['symbol'].upper() for coin in data]
    return symbols

# Function for MV Markowitz

# Covariance
def compute_covariance_matrix(price_df):
    # pct-change returns
    returns = price_df.pct_change().dropna()

    # fill missing (just in case)
    returns = returns.fillna(method="ffill").fillna(method="bfill")

    # covariance matrix
    cov_matrix = returns.cov().values

    return cov_matrix, returns


# MV (Portfolio Optimization)
def optimize_portfolio_markowitz_with_linear(
        df,
        price_df,
        investment_usd,
        risk_aversion,
        risk_tolerance,
        per_asset_max
    ):
    """
    Markowitz optimization with additional linear-risk constraint:

       maximize     μ^T w  -  λ * w^T Σ w
       subject to   sum(w_i * linear_risk_i) <= risk_tolerance
                    sum(w_i) <= 1
                    0 <= w_i <= per_asset_max

    linear_risk_i is the 'Risk' column in your portfolio_summary (std, not variance)
    """

    # Validate
    required = {"Coin","Expected Return","Risk","Action","Price"}
    if not required.issubset(df.columns):
        raise ValueError(f"DataFrame must contain: {required}")

    mu = df["Expected Return"].values.astype(float)
    linear_risk = df["Risk"].values.astype(float)   # THIS is your old risk (std)
    n = len(mu)

    # Covariance matrix
    cov_matrix = price_df.pct_change().dropna().cov().values

    # Objective func
    def objective(w):
        port_mean = mu @ w
        port_var = w @ cov_matrix @ w
        return -(port_mean - risk_aversion * port_var)

    # Constraints
    cons = []

    # 1) sum w <= 1
    cons.append({
        "type": "ineq",
        "fun": lambda w: 1.0 - np.sum(w)
    })

    # 2) linear risk: sum(w_i * linear_risk_i) <= risk_tolerance
    cons.append({
        "type": "ineq",
        "fun": lambda w: risk_tolerance - np.sum(w * linear_risk)
    })

    # Bounds
    bounds = [(0, per_asset_max) for _ in range(n)]

    # Initial guess
    x0 = np.ones(n) / n

    # Solver
    result = minimize(
        objective,
        x0,
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
    )

    if not result.success:
        raise RuntimeError(f"Optimization failed: {result.message}")

    w = result.x

    # Output & formatting
    out = df.copy().reset_index(drop=True)
    out["Weight"] = w
    out["USD Allocation"] = w * investment_usd
    out["Units (signed)"] = out["USD Allocation"] / out["Price"] * out["Action"]

    # Return and risks
    portfolio_return = float(mu @ w)
    portfolio_variance = float(w @ cov_matrix @ w)
    portfolio_std = np.sqrt(portfolio_variance)

    portfolio_linear_risk = float(np.sum(w * linear_risk))

    return out, portfolio_return, portfolio_std, portfolio_linear_risk, result

# Main

In [3]:
# Get top coins. We  use separate cell because coin gecko API

top_coins = get_top_coins()
# Future trading -> using X-USDT
i = 0
top_coins_usdt = []
while i < len(top_coins):
    top_coins_usdt.append(top_coins[i]+"-USDT")
    i = i + 1
top_coins_usdt = np.unique(top_coins_usdt)
print(top_coins_usdt) # debugging

['2Z-USDT' 'A-USDT' 'AAVE-USDT' 'AB-USDT' 'ADA-USDT' 'AERO-USDT'
 'ALGO-USDT' 'APT-USDT' 'AR-USDT' 'ARB-USDT' 'ASBNB-USDT' 'ASTER-USDT'
 'ATOM-USDT' 'AVAX-USDT' 'B-USDT' 'BAT-USDT' 'BBTC-USDT' 'BCH-USDT'
 'BDX-USDT' 'BEAT-USDT' 'BFUSD-USDT' 'BGB-USDT' 'BNB-USDT' 'BNSOL-USDT'
 'BONK-USDT' 'BORG-USDT' 'BSC-USD-USDT' 'BSV-USDT' 'BTC-USDT' 'BTC.B-USDT'
 'BTT-USDT' 'BUIDL-USDT' 'BUSD-USDT' 'CAKE-USDT' 'CBBTC-USDT' 'CBETH-USDT'
 'CC-USDT' 'CFX-USDT' 'CGETH.HASHKEY-USDT' 'CHZ-USDT' 'CLBTC-USDT'
 'CMETH-USDT' 'COMP-USDT' 'CRO-USDT' 'CRV-USDT' 'CRVUSD-USDT' 'CUSD-USDT'
 'DAI-USDT' 'DASH-USDT' 'DCR-USDT' 'DOGE-USDT' 'DOT-USDT' 'EETH-USDT'
 'ENA-USDT' 'ENS-USDT' 'ENZOBTC-USDT' 'ETC-USDT' 'ETH-USDT' 'ETHFI-USDT'
 'ETHX-USDT' 'EURC-USDT' 'EUTBL-USDT' 'EZETH-USDT' 'FARTCOIN-USDT'
 'FBTC-USDT' 'FDIT-USDT' 'FDUSD-USDT' 'FET-USDT' 'FF-USDT'
 'FIGR_HELOC-USDT' 'FIL-USDT' 'FLOKI-USDT' 'FLOW-USDT' 'FLR-USDT'
 'FOLKS-USDT' 'FRAX-USDT' 'FRXETH-USDT' 'GALA-USDT' 'GHO-USDT' 'GNO-USDT'
 'GRT-USDT' 'GT-USDT' 'G

In [4]:
# Main loop
# expected return & action set (long, short, do nothing)

backtest_threshold = 0.01 # decimal value expected return for backtest. Strictness: threshold*2
expected_return = []
risk = []
action_set = []
price = []

# Main Algorithm
print("===================================================================")
print("Running XGB strategy @", datetime.now())
print("===================================================================\n")


for coin in top_coins_usdt:
    try:

        # Fetching data
        df = fetch_data(coin)
        print("Successfull retrieving data of", coin) # debug
        df_close = df['close']
        
        # dataframe creation
        P1 = []
        P3 = []
        P7 = []
        P15 = []
        P30 = []
        r_mean = []
        r_std = []
        target = []
        
        i = 31
        while i <= 188:
            # Predictor(s)
            P1.append((df_close[i-1] - df_close[i-2])/df_close[i-2])
            P3.append((df_close[i-1] - df_close[i-4])/df_close[i-4])
            P7.append((df_close[i-1] - df_close[i-8])/df_close[i-8])
            P15.append((df_close[i-1] - df_close[i-16])/df_close[i-16])
            P30.append((df_close[i-1] - df_close[i-31])/df_close[i-31])
        
            returns = df_close[i-31:i].pct_change()
            r_mean.append(returns.mean())
            r_std.append(returns.std())
        
            # Target
            target.append((df_close[i]-df_close[i-1])/df_close[i-1])
            
            i = i + 1

        X = pd.DataFrame(
            {'P1': P1,
             'P3': P3,
             'P7': P7,
             'P15' : P15,
             'P30' : P30,
             'r_mean' : r_mean,
             'r_std' : r_std,
            })
        
        y = pd.DataFrame(
            {
             'target' : target
            })

        # Machine Learning Algorithm
        # -----------------------------
        # 1. Time-series train/test split
        # -----------------------------
        test_size = int(len(X) * 0.2)
        
        X_train = X.iloc[:-test_size]
        y_train = y.iloc[:-test_size]
        
        X_test = X.iloc[-test_size:]
        y_test = y.iloc[-test_size:]
        
        # Time-series CV
        tscv = TimeSeriesSplit(n_splits=5)
        
        # -----------------------------
        # 2. Pipeline: Scale → PCA → XGBoost
        # -----------------------------
        pipe = Pipeline([
            ("scale", RobustScaler()),
            ("pca", PCA()),
            ("model", XGBRegressor(
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            ))
        ])
        
        # -----------------------------
        # 3. Parameter grid
        # -----------------------------
        param_grid = {
            "pca__n_components": [0.90, 0.95, 0.99],
            "model__n_estimators": [100, 200],
            "model__max_depth": [3, 5, 7, 10, 25],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__subsample": [0.75, 0.8, 1.0]
        }
        
        # -----------------------------
        # 4. Grid Search using RMSE
        # -----------------------------
        grid = GridSearchCV(
            pipe,
            param_grid,
            cv=tscv,
            scoring="neg_root_mean_squared_error",  # RMSE but negated for sklearn
            n_jobs=-1,
            refit=True
        )
        
        grid.fit(X_train, y_train)
        
        # -----------------------------
        # 5. Results
        # -----------------------------
        best_params = grid.best_params_
        cv_rmse = -grid.best_score_  # convert negative RMSE → positive

        
        # -----------------------------
        # 6. Test RMSE
        # -----------------------------
        y_pred = grid.predict(X_test)
        # Step 5 and 6 only for debugging
        
        # Backtesting df creation
        P1 = []
        P3 = []
        P7 = []
        P15 = []
        P30 = []
        r_mean = []
        r_std = []
        target = []
        
        i = 189
        while i <= 198:
            
            # Predictor(s)
            P1.append((df_close[i-1] - df_close[i-2])/df_close[i-2])
            P3.append((df_close[i-1] - df_close[i-4])/df_close[i-4])
            P7.append((df_close[i-1] - df_close[i-8])/df_close[i-8])
            P15.append((df_close[i-1] - df_close[i-16])/df_close[i-16])
            P30.append((df_close[i-1] - df_close[i-31])/df_close[i-31])
        
            returns = df_close[i-30:i].pct_change()
            r_mean.append(returns.mean())
            r_std.append(returns.std())
        
            # Target
            target.append((df_close[i]-df_close[i-1])/df_close[i-1])
            
            i = i + 1
        
        X_backtest = pd.DataFrame(
            {'P1': P1,
             'P3': P3,
             'P7': P7,
             'P15' : P15,
             'P30' : P30,
             'r_mean' : r_mean,
             'r_std' : r_std,
            })
        y_backtest = pd.DataFrame(
            {
             'target' : target
            })

        # ---------- Backtest Prediction ----------
        y_backtest_pred = grid.predict(X_backtest)
        # Action taken
        action = []
        profit = []
        k = 0
        while k < len(y_backtest_pred):
            if y_backtest_pred[k] >= backtest_threshold: # threshold
                action.append(1)
            elif y_backtest_pred[k] <= -backtest_threshold:
                action.append(-1)
            else:
                action.append(0)
            profit.append(action[-1]*target[k])
            k = k + 1

        expected_return.append(np.mean(profit))
        risk.append(np.std(profit))


        # prediction for today
        i = 199
        P1 = [(df_close[i-1] - df_close[i-2])/df_close[i-2]]
        P3 = [(df_close[i-1] - df_close[i-4])/df_close[i-4]]
        P7 = [(df_close[i-1] - df_close[i-8])/df_close[i-8]]
        P15 = [(df_close[i-1] - df_close[i-16])/df_close[i-16]]
        P30 = [(df_close[i-1] - df_close[i-31])/df_close[i-31]]
        returns = df_close[i-30:i].pct_change()
        r_mean = [returns.mean()]
        r_std = [returns.std()]
        X_today = pd.DataFrame(
            {'P1': P1,
             'P3': P3,
             'P7': P7,
             'P15' : P15,
             'P30' : P30,
             'r_mean' : r_mean,
             'r_std' : r_std,
            })
        y_today_pred = grid.predict(X_today)
        if y_today_pred[0] >= backtest_threshold: # threshold
            action_set.append(1)
        elif y_today_pred[0] <= -backtest_threshold:
            action_set.append(-1)
        else:
            action_set.append(0)
        price.append(df_close[198]) # i = 199, so this should be i-1 since we're using yesterday's close for reference to get return of today's close
        print("Done analyzing", coin) # debug end
        print("\n")
    
    except Exception as e:
        print(f"Skipping {coin} due to error: {e}\n")
        action_set.append(0)
        expected_return.append(0)
        risk.append(0)
        price.append(0)
        continue

Running XGB strategy @ 2025-12-14 07:01:10.145348

Successfull retrieving data of 2Z-USDT
Skipping 2Z-USDT due to error: 74

Successfull retrieving data of A-USDT
Done analyzing A-USDT


Successfull retrieving data of AAVE-USDT
Done analyzing AAVE-USDT


Skipping AB-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=AB-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Successfull retrieving data of ADA-USDT
Done analyzing ADA-USDT


Successfull retrieving data of AERO-USDT
Done analyzing AERO-USDT


Successfull retrieving data of ALGO-USDT
Done analyzing ALGO-USDT


Successfull retrieving data of APT-USDT
Done analyzing APT-USDT


Successfull retrieving data of AR-USDT
Done analyzing AR-USDT


Successfull retrieving data of ARB-USDT
Done analyzing ARB-USDT


Skipping ASBNB-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/

C:\Users\Rafi\anaconda3\lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Done analyzing TUSD-USDT


Successfull retrieving data of TWT-USDT
Done analyzing TWT-USDT


Skipping UBTC-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=UBTC-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Skipping UDS-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=UDS-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Skipping ULTIMA-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=ULTIMA-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Successfull retrieving data of UNI-USDT
Done analyzing UNI-USDT


Skipping USD0-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrumen

C:\Users\Rafi\anaconda3\lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Done analyzing USDE-USDT


Skipping USDF-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=USDF-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Skipping USDG-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=USDG-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Skipping USDS-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=USDS-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Skipping USDT-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=USDT-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Skipping USDT0-USDT due to error: 404 Client Erro

...

C:\Users\Rafi\anaconda3\lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Done analyzing XLM-USDT


Skipping XMR-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=XMR-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Successfull retrieving data of XPL-USDT
Skipping XPL-USDT due to error: 81

Successfull retrieving data of XRP-USDT
Done analyzing XRP-USDT


Successfull retrieving data of XTZ-USDT


C:\Users\Rafi\anaconda3\lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Done analyzing XTZ-USDT


Skipping ZBCN-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=ZBCN-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Skipping ZEC-USDT due to error: 404 Client Error: Not Found for url: https://data-api.coindesk.com/spot/v1/historical/days?market=bybit&instrument=ZEC-USDT&limit=200&aggregate=1&fill=true&apply_mapping=true&response_format=JSON

Successfull retrieving data of ZK-USDT
Done analyzing ZK-USDT


Successfull retrieving data of ZRO-USDT
Done analyzing ZRO-USDT




C:\Users\Rafi\anaconda3\lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


# Action Report

In [5]:
# creating summary
action_set_LS = []
i = 0
while i < len(action_set):
    if action_set[i] == 1:
        action_set_LS.append('LONG')
    elif action_set[i] == 0:
        action_set_LS.append('-')
    else:
        action_set_LS.append('SHORT')
    i = i + 1

summary = pd.DataFrame(
    {'Coin': top_coins_usdt,
     'Action': action_set,
     'Action (L/S)': action_set_LS,
     'Expected Return': expected_return,
     'Risk' : risk,
     'Price' : price,
    })

summary = summary.drop_duplicates(subset="Coin", keep="first")
sorted_summary = summary.sort_values(by=['Expected Return'], ascending = False)
sorted_summary = sorted_summary.reset_index(drop=True)
portfolio_summary = sorted_summary[(sorted_summary['Action'] != 0) & (sorted_summary['Expected Return'] > 0)]
portfolio_summary = portfolio_summary.reset_index(drop=True)
print(portfolio_summary)

          Coin  Action Action (L/S)  Expected Return      Risk        Price
0    MERL-USDT       1         LONG         0.029977  0.060002     0.454400
1     FET-USDT       1         LONG         0.022481  0.032388     0.245600
2    FRAX-USDT      -1        SHORT         0.017543  0.020907     0.708000
3    PUMP-USDT      -1        SHORT         0.012189  0.042217     0.002795
4    NEAR-USDT       1         LONG         0.011804  0.026768     1.665000
5    ONDO-USDT       1         LONG         0.011300  0.025768     0.463400
6     SEI-USDT       1         LONG         0.011236  0.037684     0.129400
7     SPX-USDT      -1        SHORT         0.009375  0.023249     0.591000
8     DOT-USDT       1         LONG         0.008515  0.035708     2.038000
9     ADA-USDT       1         LONG         0.007257  0.012240     0.410600
10   METH-USDT       1         LONG         0.007017  0.012855  3340.740000
11  ETHFI-USDT       1         LONG         0.006571  0.031235     0.808000
12     OP-US

# Portfolio Optimization

In [6]:
# Making covariance matrix
price_df = pd.DataFrame()

for coin in portfolio_summary['Coin']:

    df = fetch_data_short(coin)
    if df.empty:
        print(f"No data for {coin}, skipping.")
        continue

    # close price only, rename column to coin
    close_series = df['close'].rename(coin)

    # merge
    price_df = price_df.join(close_series, how='outer')

# Handle missing values
price_df = price_df.ffill().bfill()      # forward/backward fill
price_df = price_df.fillna(price_df.mean())  # final fallback
price_df = price_df.iloc[:-1]

print(price_df)

    MERL-USDT  FET-USDT  FRAX-USDT  PUMP-USDT  NEAR-USDT  ONDO-USDT  SEI-USDT  \
0     0.37080    0.2594      0.836   0.002889      1.851     0.5094    0.1353   
1     0.35035    0.2557      0.830   0.002948      1.806     0.5037    0.1370   
2     0.44289    0.2312      0.784   0.002789      1.627     0.4628    0.1231   
3     0.32947    0.2557      0.825   0.003262      1.803     0.5015    0.1378   
4     0.34391    0.2612      0.847   0.003166      1.847     0.5131    0.1402   
5     0.37825    0.2494      0.815   0.003118      1.797     0.4912    0.1354   
6     0.36165    0.2346      0.791   0.002837      1.681     0.4637    0.1266   
7     0.36101    0.2353      0.801   0.002983      1.708     0.4679    0.1285   
8     0.35083    0.2361      0.783   0.002916      1.701     0.4561    0.1274   
9     0.34430    0.2404      0.775   0.003056      1.744     0.4848    0.1319   
10    0.34520    0.2635      0.783   0.003067      1.811     0.4993    0.1383   
11    0.34040    0.2526     

In [7]:
# Public Use

print(f"=============================== PUBLIC USE ===============================\n")
investment_usd = 100
leverage = 1
per_asset_max = 0.4

risk_tolerance = [0.01, 0.02, 0.025, 0.05]   # linear risk constraint
risk_aversions = [0.1, 5]                  # Markowitz λ values

for dummy_lam in risk_aversions:
    print(f"~~~~~~~~~~~~~~~~~~~~~~~~~~~ Risk Aversion: {dummy_lam} ~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    for dummy_risk in risk_tolerance:
        try:
            opt_df, port_ret, port_std, port_lin_risk, res = \
                optimize_portfolio_markowitz_with_linear(
                    portfolio_summary,
                    price_df,
                    investment_usd * leverage,
                    dummy_lam,
                    dummy_risk,
                    per_asset_max
                )

            filtered = opt_df[opt_df["USD Allocation"] > 0.001].reset_index()
            print("\n==========================================================================")
            print(f"Porfolio Risk (%)    : {leverage*dummy_risk*100}")
            print(f"Return Range (%)     : {investment_usd * leverage*(port_ret-port_std):.4f} to {investment_usd * leverage*(port_ret+port_std):.4f}")
            filtered["Take Profit (%)"] = 100 * leverage * (filtered["Expected Return"] + filtered["Risk"])
            filtered["Stop Loss (%)"]  = 100 * leverage * filtered["Risk"]

            filtered.rename(columns={'USD Allocation': 'Allocation (%)', 'Action (L/S)': 'Actions', 'Stop Loss (%)': 'SL (%)','Take Profit (%)': 'TP (%)' }, inplace=True)
            print(filtered[["Coin","Actions","Allocation (%)","Price","SL (%)","TP (%)"]])

        except Exception as e:
            print(f"[Skipping] risk={dummy_risk}, lam={dummy_lam} → {e}")
            continue
    print("\n")

=============================== PUBLIC USE ===============================

~~~~~~~~~~~~~~~~~~~~~~~~~~~ Risk Aversion: 0.1 ~~~~~~~~~~~~~~~~~~~~~~~~~~~

Porfolio Risk (%)    : 1.0
Return Range (%)     : -0.6478 to 2.2785
        Coin Actions  Allocation (%)   Price    SL (%)    TP (%)
0   FET-USDT    LONG        5.055115  0.2456  3.238774  5.486873
1  FRAX-USDT   SHORT       40.000000  0.7080  2.090691  3.844977

Porfolio Risk (%)    : 2.0
Return Range (%)     : -1.5716 to 4.5906
        Coin Actions  Allocation (%)   Price    SL (%)    TP (%)
0   FET-USDT    LONG       35.930994  0.2456  3.238774  5.486873
1  FRAX-USDT   SHORT       40.000000  0.7080  2.090691  3.844977

Porfolio Risk (%)    : 2.5
Return Range (%)     : -2.0031 to 5.6006
        Coin Actions  Allocation (%)      Price    SL (%)    TP (%)
0  MERL-USDT    LONG        3.017439     0.4544  6.000165  8.997849
1   FET-USDT    LONG       40.000000     0.2456  3.238774  5.486873
2  FRAX-USDT   SHORT       40.000000     0.7080 

In [8]:
# PERSONAL USE
print(f"====================== PERSONAL USE ======================\n")
investment_usd = 25
leverage = 10
per_asset_max = 0.4

risk_tolerance = [0.01, 0.02, 0.025, 0.05, 0.08, 0.1]   # linear risk constraint
risk_aversions = [0.1, 1, 5, 10, 20, 50, 100]                  # Markowitz λ values

for dummy_risk in risk_tolerance:
    print(f"~~~~~~~~~~~ Portfolio risk: {leverage*dummy_risk*100} (%) | {leverage*dummy_risk*investment_usd} (USD) ~~~~~~~~~~~")
    for dummy_lam in risk_aversions:
        try:
            opt_df, port_ret, port_std, port_lin_risk, res = \
                optimize_portfolio_markowitz_with_linear(
                    portfolio_summary,
                    price_df,
                    investment_usd * leverage,
                    dummy_lam,
                    dummy_risk,
                    per_asset_max
                )

            filtered = opt_df[opt_df["USD Allocation"] > 0.001].reset_index()
            print("\n============================================================")
            print(f"Risk Aversion (λ)    : {dummy_lam}")
            print(f"Target Profit (USD)  : {port_ret * investment_usd * leverage:.4f}")
            print(f"Return Range (USD)   : {investment_usd * leverage*(port_ret-port_std):.4f} to {investment_usd * leverage*(port_ret+port_std):.4f}")
            filtered["Take Profit (%)"] = 100 * leverage * (filtered["Expected Return"] + filtered["Risk"])
            filtered["Stop Loss (%)"]  = 100 * leverage * filtered["Risk"]

            filtered.rename(columns={'USD Allocation': 'Allocation (USD)', 'Action (L/S)': 'Actions', 'Stop Loss (%)': 'SL (%)','Take Profit (%)': 'TP (%)'}, inplace=True)
            print(filtered[["Coin","Actions","Allocation (USD)","Price","SL (%)","TP (%)"]])

        except Exception as e:
            print(f"[Skipping] risk={dummy_risk}, lam={dummy_lam} → {e}")
            continue
    print("\n")

====================== PERSONAL USE ======================

~~~~~~~~~~~ Portfolio risk: 10.0 (%) | 2.5 (USD) ~~~~~~~~~~~

Risk Aversion (λ)    : 0.1
Target Profit (USD)  : 2.0384
Return Range (USD)   : -1.6195 to 5.6962
        Coin Actions  Allocation (USD)   Price     SL (%)     TP (%)
0   FET-USDT    LONG         12.637788  0.2456  32.387742  54.868728
1  FRAX-USDT   SHORT        100.000000  0.7080  20.906906  38.449770

Risk Aversion (λ)    : 1
Target Profit (USD)  : 2.0384
Return Range (USD)   : -1.6195 to 5.6962
        Coin Actions  Allocation (USD)   Price     SL (%)     TP (%)
0   FET-USDT    LONG         12.637788  0.2456  32.387742  54.868728
1  FRAX-USDT   SHORT        100.000000  0.7080  20.906906  38.449770

Risk Aversion (λ)    : 5
Target Profit (USD)  : 1.9588
Return Range (USD)   : -0.6417 to 4.5592
        Coin Actions  Allocation (USD)   Price     SL (%)     TP (%)
0  MERL-USDT    LONG          6.821636  0.4544  60.001651  89.978494
1  FRAX-USDT   SHORT        100.00

# Note

There are one important input for this model, that is 'backtest_threshold' value. The bigger the value means that we have more strict filter.
Total inputs are:
1. fetching data : daily, hourly, 4 hours, etc
2. how many top coins we want to run in our ML
3. backtest_threshold (IMPORTANT)
4. investment_usd
5. risk_tolerance
6. per_asset_max : this one is for preventing our portfolio to only contain one asset (100% allocation). There are 2 of this, in def and driver~
7. risk_aversion
   0.1 -> very high risk
   1   -> balanced risk - return
   5   -> minor safety
   10  -> medium safety
   100 -> very high safety
   Bigger means less risk

Suggested running time: shortly after 7 AM (UTC + 7). The coindesk server is at 00.00 at that time so the data is fresh!
Only use max 80% of total asset since there is a MARGIN to consider. Example: if you have 50 usd then only use 40 usd for investment_usd variable.